1、不同的文档，使用不同的文档加载器 （熟悉）

txt文档：TextLoader

pdf文档：PyPDFLoader

csv文档：CSVLoader

json文档：JSONLoader

html文档：UnstructuredHTMLLoader

md文档：UnStructuredMarkdownLoader

文件目录：DirectoryLoader


2、创建好XxxLoader的实例以后，都需要调用load()，在内存中返回一个list[Document]







# 1、加载Txt文档



In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader

# 指明txt文档的路径
file_path = "./asset/load/01-langchain-utf-8.txt"

# 创建一个TextLoader的实例
text_loader = TextLoader(
    file_path=file_path,
    encoding="utf-8",
)

# 调用load()，返回一个list[Document]
docs = text_loader.load()


In [ ]:
print(docs)

print(len(docs))  # 查看列表中元素的个数‘

print(docs[0])

In [ ]:
# 显示Document对象的元数据
print(docs[0].metadata)

# 显示文档中的内容信息
print(docs[0].page_content)

In [ ]:
from langchain_community.document_loaders import TextLoader

# 指明txt文档的路径
file_path = "./asset/load/01-langchain-gbk.txt"

# 创建一个TextLoader的实例
text_loader = TextLoader(
    file_path=file_path,
    encoding="gbk",  #此时使用的解码集一定要与当初存储此文件使用的编码集相同，否则报错！
)

# 调用load()，返回一个list[Document]
docs = text_loader.load()

print(docs)


# 2、加载pdf文档


In [ ]:
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader(
    file_path="./asset/load/02-load.pdf"
)

docs = pdf_loader.load()

print(docs)

print(len(docs))

也可以加载网络中的一个文件：

In [ ]:
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader(
    file_path="https://arxiv.org/pdf/2302.03803"
)

docs = pdf_loader.load()

print(len(docs))

for doc in docs:
    print(doc)

# 3、加载CSV文档

In [ ]:
from langchain_community.document_loaders import CSVLoader

csv_loader = CSVLoader(
    file_path="./asset/load/03-load.csv"
)

docs = csv_loader.load()

print(len(docs))


print(docs)

for doc in docs:
    print(doc)

In [ ]:
from langchain_community.document_loaders import CSVLoader

csv_loader = CSVLoader(
    file_path="./asset/load/03-load.csv",
    source_column="author"
)

docs = csv_loader.load()

for doc in docs:
    print(doc)

# 4、加载JSON文档

举例1：加载json文件中的所有的数据

In [ ]:
from langchain_community.document_loaders import JSONLoader

json_loader = JSONLoader(
    file_path="./asset/load/04-load.json",
    jq_schema=".", #表示加载所有的字段
    text_content=False, #将加载的json对象转换为json字符串
)

docs = json_loader.load()

print(docs)

举例2：加载json文件中messages[]中的所有的content字段

In [ ]:
from langchain_community.document_loaders import JSONLoader

json_loader = JSONLoader(
    file_path="./asset/load/04-load.json",
    jq_schema=".messages[].content", #加载messages[]的所有的content字段
    #text_content=False, #将加载的json对象转换为json字符串
)

docs = json_loader.load()

for doc in docs:
    print(doc.page_content)

举例3：提取04-response.json文件中嵌套在 data.items[].content 的文本

In [ ]:
from langchain_community.document_loaders import JSONLoader

# 方式1：
# json_loader = JSONLoader(
#     file_path="./asset/load/04-response.json",
#     jq_schema=".data.items[].content", #data.items[].content
# )

# 方式2：
json_loader = JSONLoader(
    file_path="./asset/load/04-response.json",
    jq_schema=".data.items[]", #data.items[].content
    content_key=".content",
    is_content_key_jq_parsable=True, #用jq解析content_key
)

docs = json_loader.load()

for doc in docs:
    print(doc.page_content)

举例4：提取04-response.json文件中嵌套在 data.items[] 里的 title、content 和 其文本



In [ ]:
# 1.导入相关依赖
from langchain_community.document_loaders import JSONLoader
from pprint import pprint

# 2.定义json文件的路径
file_path = 'asset/load/04-response.json'

# 3.定义JSONLoader对象
# 提取嵌套在 data.items[].content 的文本，并保留其他字段作为元数据
# loader = JSONLoader(
#     file_path=file_path,
#     # jq_schema=".data.items[] | {id, author, text: (.title + '\n' + .content)}",
#     jq_schema='''.data.items[] | {
#     id,
#     author,
#     created_at,
#     title, # 保留title字段
#     text: (.title + "\n" + .content)
#     }''',
#     content_key=".text",  # 再从条目中提取 content 字段
#     is_content_key_jq_parsable=True  # 用jq解析content_key
# )
loader = JSONLoader(
    file_path=file_path,
    # jq_schema=".data.items[] | {id, author, text: (.title + '\n' + .content)}",
    jq_schema=".data.items[]",
    content_key='.title + "\\n\\n" + .content',
    is_content_key_jq_parsable=True  # 用jq解析content_key
)

# loader = JSONLoader(
#     file_path=file_path,
#     # jq_schema=".data.items[] | {id, author, text: (.title + '\n' + .content)}",
#     jq_schema='''
#         .data.items[] | {
#             metadata: {
#                 id,
#                 author,
#                 created_at
#             },
#             content: (.title + "\n\n" + .content)
#         }
#     ''',  # 构建新结构
#      content_key='.title + "\\n\\n" + .content',
#     is_content_key_jq_parsable=True  # 用jq解析content_key
# )

# 4.加载
data = loader.load()

for doc in data:
    print(doc.page_content)

# 5、加载HTML文档

In [ ]:
# 1.导入相关的依赖
from langchain.document_loaders import UnstructuredHTMLLoader

# 2.定义UnstructuredHTMLLoader对象
# strategy:
#   "fast" 解析加载html文件速度是比较快（但可能丢失部分结构或元数据）
#   "hi_res": (高分辨率解析) 解析精准（速度慢一些）
#   "ocr_only"  强制使用ocr提取文本，仅仅适用于图像（对HTML无效）

# mode ：one of `{'paged', 'elements', 'single'}
#    "elements"  按语义元素（标题、段落、列表、表格等）拆分成多个独立的小文档

html_loader = UnstructuredHTMLLoader(
    file_path="asset/load/05-load.html",
    mode="elements",
    strategy="fast"
)

# 3.加载
docs = html_loader.load()

print(len(docs))  # 16

# 4.打印
for doc in docs:
    print(doc)


# 6、加载markdown文档

In [ ]:
# 1.导入相关的依赖
from langchain.document_loaders import UnstructuredMarkdownLoader
from pprint import pprint

# 2.定义UnstructuredMarkdownLoader对象
md_loader = UnstructuredMarkdownLoader(
    file_path="asset/load/06-load.md",
    strategy="fast"
)

# 3.加载
docs = md_loader.load()

print(len(docs))
# 4.打印
for doc in docs:
    pprint(doc)

In [ ]:
# 1.导入相关的依赖
from langchain.document_loaders import UnstructuredMarkdownLoader
from pprint import pprint

# 2.定义UnstructuredMarkdownLoader对象
md_loader = UnstructuredMarkdownLoader(
    file_path="asset/load/06-load.md",
    strategy="fast",
    mode="elements"
)

# 3.加载
docs = md_loader.load()

print(len(docs))
# 4.打印
for doc in docs:
    pprint(doc)

# 7、加载指定的文件目录


In [ ]:
# 1.导入相关的依赖
from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import PythonLoader
from pprint import pprint

# 2.定义DirectoryLoader对象,指定要加载的文件夹路径、要加载的文件类型和是否使用多线程
directory_loader = DirectoryLoader(
    path="./asset/load",
    glob="*.py",
    use_multithreading=True,
    show_progress=True,
    loader_cls=PythonLoader
)

# 3.加载
docs = directory_loader.load()

# 4.打印
print(len(docs))
for doc in docs:
    pprint(doc)